# Digital Cash Management Forecast: lab (data scientist)

Explore the data and try models here without touching production: nothing in this notebook changes the flow, the
saved models or the production model.

1. **EDA**: the training rows by category, totals over time, seasonality, and what the models learn.
2. **Models to try**: several forecasters trained and tested side by side on the same test as champion / challenger
   (trained without the last 6 months before the cut-off, tested on them), next to the production model and the
   simple guess (the average of the last 3 months), with the decision champion / challenger would take for each.
   Each try is recorded in Experiment Tracking, experiment `cashflow_lab` (Lab → Experiment tracking); lab tries
   are never deployed.
3. **Put one in production.**

**Kernel:** the code env `cashflow_forecast`. If it is not in the kernel list, ask whoever manages it to enable its
Jupyter support; meanwhile, the recipe `try_forecaster` tries one forecaster without a notebook (set the project
variable `lab_forecaster` and build `lab_results`, Lab zone).

In the sandbox the data is simulated: there, the scores show that the code works, not which model is better.

In [ ]:
import calendar
import importlib
import logging
import warnings

warnings.filterwarnings("ignore", category=FutureWarning, module="google")      # third-party notices on import
logging.getLogger("matplotlib").setLevel(logging.ERROR)
try:
    import catboost
    import mlflow
except ImportError:
    raise RuntimeError("Use the kernel of the code env cashflow_forecast (Kernel > Change kernel). If it is not in the "
                       "list, ask whoever manages that code env to enable its Jupyter support; meanwhile, set the "
                       "project variable lab_forecaster and build lab_results (Lab zone).")
import dataiku

warnings.filterwarnings("ignore", category=DeprecationWarning)    # third-party notices; after dataiku, which shows them
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

import cashflow
import models
import tracking

for module in (cashflow, models, tracking):     # pick up changes to the project library without a restart
    importlib.reload(module)
from cashflow import CATEGORIES, read_dataset
from models import CAT_FEATURES, CatBoostForecaster, lab_compare, model_matrix

pd.set_option("display.max_columns", 30)
names = {code: name for code, (name, _, _) in CATEGORIES.items()}
rows = read_dataset("training_rows")
print(f"training_rows: {len(rows):,} rows, {rows['client_no'].nunique()} clients, cut-off months "
      f"{rows['cutoff_month'].min():%Y-%m} to {rows['cutoff_month'].max():%Y-%m}")

## 1. EDA

The training rows: one row per client, category, cut-off month and months ahead (1 to 3). The models predict
`target`: the month's amount (`actual`) minus the part already scheduled at the cut-off (`obligations_known`), which
the forecast adds back.

- **zero_months**: the share of months with no movement. Lumpy categories are hard to forecast.
- **known_at_cutoff**: the share of the amount already scheduled one month ahead. That part is not predicted.
- **volatility**: how much a client's month varies (median coefficient of variation of the monthly amounts).

In [ ]:
# Each month once per client and category (months ahead 1): the amount (actual), the part already scheduled at the
# cut-off (obligations_known) and what the models predict (target = actual - obligations_known). The net cash flow
# (money in minus money out) is left out here: its ratios mix signs.
monthly = rows[(rows["months_ahead"] == 1) & (rows["category"] != "NET")].assign(
    category_name=lambda d: d["category"].map(names))
by_category = monthly.groupby("category_name")
series = monthly.groupby(["category_name", "client_no"])["actual"]
volatility = (series.std() / series.mean().abs()).replace(np.inf, np.nan)
overview = pd.DataFrame({
    "clients": by_category["client_no"].nunique(),
    "months": by_category.size(),
    "average_month_sar": by_category["actual"].mean().round(0),
    "zero_months": by_category["actual"].apply(lambda s: (s.abs() < 1).mean()).round(3),
    "known_at_cutoff": (by_category["obligations_known"].sum() / by_category["actual"].sum()).round(3),
    "volatility": volatility.groupby(level="category_name").median().round(2),
}).sort_values("average_month_sar", ascending=False)
display(overview)
print("Clients by segment:", rows.drop_duplicates("client_no")["segment"].value_counts().to_dict())

Totals over time and seasonality. Ramadan and Eid move about 11 days earlier each year, so a calendar-month profile
blurs them.

In [ ]:
# Monthly totals over all clients, money in and money out by category
fig, axes = plt.subplots(1, 2, figsize=(15, 4.5))
for direction, ax in zip(("IN", "OUT"), axes):
    totals = monthly[monthly["direction"] == direction].pivot_table(
        index="target_month", columns="category_name", values="actual", aggfunc="sum")
    (totals / 1e6).plot(ax=ax, title=f"Money {direction.lower()} per month, all clients (million SAR)")
    ax.set_xlabel("")
    ax.legend(fontsize=7)
plt.tight_layout()
plt.show()

# Seasonality without the trend: all clients' amount in a calendar month divided by their average of the 12 months
# before it (1 = a usual month). Totals, not a median: most months of lumpy categories are zero.
seasonal = monthly.assign(calendar_month=monthly["target_month"].dt.month).pivot_table(
    index="category_name", columns="calendar_month", values=["actual", "avg_12m"], aggfunc="sum")
profile = seasonal["actual"] / seasonal["avg_12m"]
fig, ax = plt.subplots(figsize=(10, 4.5))
image = ax.imshow(profile.values, cmap="RdBu_r", vmin=0.5, vmax=1.5, aspect="auto")
ax.set_xticks(range(len(profile.columns)))
ax.set_xticklabels([calendar.month_abbr[m] for m in profile.columns])
ax.set_yticks(range(len(profile.index)))
ax.set_yticklabels(profile.index)
ax.set_title("Seasonality: month / the 12 months before, all clients (1 = usual)")
fig.colorbar(image, ax=ax)
plt.show()

What the models learn: `target` divided by the series scale (the client's usual amount in that category), its spread
per category, and how much each input moves with it.

In [ ]:
# What the models learn: target / series_scale (the client's usual amount in that category), per category
scaled = rows.assign(category_name=rows["category"].map(names), y=rows["target"] / rows["series_scale"])
display(scaled.groupby("category_name")["y"].describe(percentiles=[0.01, 0.1, 0.5, 0.9, 0.99]).round(2))

# How much each input moves with it (Spearman correlation, inputs divided by the series scale too)
inputs = ["last_amount", "avg_3m", "same_month_last_year", "free_last", "free_avg_3m", "oblig_target_month"]
correlation = {}
for name, part in scaled.groupby("category_name"):
    x = part[inputs].div(part["series_scale"], axis=0)
    x = x.loc[:, x.nunique() > 1]                   # an input that never changes has no correlation (blank)
    correlation[name] = x.corrwith(part["y"], method="spearman")
display(pd.DataFrame(correlation).T.reindex(columns=inputs).round(2))

## 2. Models to try

A forecaster is a class with `name`, `params` (its settings), `inputs` (the model-row columns it reads), `fit(rows)`
and `predict(rows)`, which returns `p10`, `ev` and `p90` in SAR (details in `models.py`, project library). Three
examples, next to the reference CatBoost forecaster (`catboost`, the production recipe):

- `simple_guess`: no model, the average of the last 3 months, with a band from its own past errors
- `catboost_depth8`: the reference CatBoost forecaster with deeper trees
- `sklearn_boosting`: scikit-learn gradient boosting, one model per output (EV, P10, P90)

Change them or add your own, then run the comparison. CatBoost, scikit-learn, pandas and matplotlib are in the code
env; other packages need the admin.

In [ ]:
class SimpleGuess:
    """The simple guess as a forecaster: EV = average of the last 3 months (without the scheduled part); P10 and P90 =
    EV plus the errors it made in training at 10% and 90%, per category, scaled by the series scale."""
    name = "simple_guess"
    params = {"quantiles": "0.1,0.9"}
    inputs = ["category", "free_avg_3m", "series_scale"]

    def _guess(self, rows):
        return np.where(rows["category"] == "NET", rows["free_avg_3m"], rows["free_avg_3m"].clip(lower=0))

    def fit(self, rows):
        error = (rows["target"] - self._guess(rows)) / rows["series_scale"]
        self.bands = error.groupby(rows["category"]).quantile([0.1, 0.9]).unstack()

    def predict(self, rows):
        ev = self._guess(rows)
        band = self.bands.reindex(rows["category"]).fillna(0.0).values
        scale = rows["series_scale"].values
        return pd.DataFrame({"p10": ev + band[:, 0] * scale, "ev": ev, "p90": ev + band[:, 1] * scale},
                            index=rows.index)


class CatBoostDepth8(CatBoostForecaster):
    """The reference CatBoost forecaster with deeper trees."""
    name = "catboost_depth8"
    params = {**CatBoostForecaster.params, "depth": 8}


class SklearnBoosting:
    """scikit-learn gradient boosting (HistGradientBoostingRegressor): one model for the EV, one for P10 and one for
    P90 (quantile loss), with the reference forecaster's inputs and scaling (amounts / series scale)."""
    name = "sklearn_boosting"
    params = {"max_iter": 400, "learning_rate": 0.05, "max_leaf_nodes": 31, "min_samples_leaf": 40, "random_state": 0}
    inputs = CatBoostForecaster.inputs

    def _matrix(self, rows):
        X = model_matrix(rows)
        for c in CAT_FEATURES:            # categories as numbers; one not seen in training counts as missing
            X[c] = X[c].map(self.codes[c]).astype(float)
        return X

    def fit(self, rows):
        from sklearn.ensemble import HistGradientBoostingRegressor
        self.codes = {c: {v: i for i, v in enumerate(sorted(values.unique()))}
                      for c, values in model_matrix(rows)[CAT_FEATURES].items()}
        X, y = self._matrix(rows), rows["target"] / rows["series_scale"]
        is_category = [c in CAT_FEATURES for c in X.columns]
        losses = {"ev": {"loss": "squared_error"}, "p10": {"loss": "quantile", "quantile": 0.1},
                  "p90": {"loss": "quantile", "quantile": 0.9}}
        self.models = {k: HistGradientBoostingRegressor(categorical_features=is_category, **loss, **self.params).fit(X, y)
                       for k, loss in losses.items()}

    def predict(self, rows):
        X, scale = self._matrix(rows), rows["series_scale"].values
        return pd.DataFrame({k: m.predict(X) * scale for k, m in self.models.items()}, index=rows.index)

In [ ]:
# Train each on the same months, test it on the last 6, next to the production model. catboost is the production
# recipe trained again: it should match the production model's scores (a check that the test is the same).
RECORD = True       # record each try in Experiment Tracking (experiment cashflow_lab)
result = lab_compare([SimpleGuess(), CatBoostForecaster(), CatBoostDepth8(), SklearnBoosting()], record=RECORD)
print("All categories, all months ahead. EV error: lower is better. P10-P90 coverage: aim 80%.")
display(result["board"])
print("EV error per category:")
display(result["errors"])
print("P10-P90 coverage per category:")
display(result["coverage"])
print("EV error per months ahead:")
display(result["by_months_ahead"])

In [ ]:
errors = result["errors"].drop(index="All categories")
if "simple_guess" in errors:                         # the same EV as the baseline: shown once
    errors = errors.drop(columns="simple_guess_baseline")
ax = errors.plot.barh(figsize=(10, 8), title="EV error per category (lower is better)")
ax.set_xlim(0, min(errors.max().max() * 1.05, 3))
ax.set_ylabel("")
ax.invert_yaxis()
plt.tight_layout()
plt.show()

## 3. Put one in production

1. Copy its class into `forecasters.py` (project library) with the imports it needs (for example
   `import numpy as np`, `import pandas as pd`, `from models import CAT_FEATURES, CatBoostForecaster, model_matrix`)
   and add it to `FORECASTERS`. The setup notebook never overwrites that file.
2. Set the project variable `forecaster` to its name.
3. Run the scenario **Retrain, test and promote**. Champion / challenger promotes it only if it beats the
   production model on the same 6 months; the next Nightly batch forecasts with it.
4. To stop training it, set `forecaster` back to `catboost`. To use an earlier model now, activate its version in
   the three saved models.